In [9]:
import pandas as pd
import numpy as np
import random
import time

# Read input data
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/3-unit.xlsx',
    sheet_name='Table 1'
)
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd3u.xlsx')
power_demand = power_demand_data['load'].values

# Define Dream Optimization Algorithm_DOA-based power calculation
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    pop=30, max_iter=100):
    dim = len(pmin)
    lb, ub = pmin, pmax

    # Objective: Fuel cost + penalty for demand mismatch
    def fobj(pos):
        fuel = np.sum(a + b * pos + c * pos**2)
        penalty = abs(demand - np.sum(pos)) * 10000
        return fuel + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            # choose k with safe randint bounds
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))  # inclusive upper bound
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        # Boundary handling
                        if x[j, h] > ub[h] or x[j, h] < lb[h]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h] = x[sel, h]
                            else:
                                x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        # Local exploitation around sbest
        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            # safe randint when km could be 2
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                if x[j, h] > ub[h] or x[j, h] < lb[h]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h] = x[sel, h]
                    else:
                        x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment (relative to x[0] as in your original code)
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# Fuel cost calculation
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP)))
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []  # kept name per your original code
summary_rows = []  # NEW: clean summary rows for Excel

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    # Keep all_outputs for later ramp-rate computation (don't try to write it directly to Excel)
    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P),
        'Total Fuel Cost': total_Fuel_Cost,
        'Computation Time': end_time - start_time
    })

    # Clean summary row (serializable)
    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp)': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': iterations
    })

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # kept as-is (total cost assigned to each unit)
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# Save results to Excel (write clean tables only)
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED) 3u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED) 3u.xlsx'")

Load Counter: 1
Output for Power Demand 850.00 MW
   Unit  Power Produced (MW)
0     1           422.201560
1     2           243.587776
2     3           184.210664
Total Power Produced: 850.00 MW
Total Fuel Cost: Rp 8230.262092981664
Computation Time: 0.0371708869934082 seconds
------------------------------------------------

All results saved to 'DOA (ED) 3u.xlsx'


In [11]:
import pandas as pd
import numpy as np
import random
import time

# Read input data
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/6-unit.xlsx',
    sheet_name='Table 1'
)
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd6u.xlsx')
power_demand = power_demand_data['load'].values

# Define Dream Optimization Algorithm_DOA-based power calculation
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    pop=30, max_iter=100):
    dim = len(pmin)
    lb, ub = pmin, pmax

    # Objective: Fuel cost + penalty for demand mismatch
    def fobj(pos):
        fuel = np.sum(a + b * pos + c * pos**2)
        penalty = abs(demand - np.sum(pos)) * 10000
        return fuel + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            # choose k with safe randint bounds
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))  # inclusive upper bound
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        # Boundary handling
                        if x[j, h] > ub[h] or x[j, h] < lb[h]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h] = x[sel, h]
                            else:
                                x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        # Local exploitation around sbest
        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            # safe randint when km could be 2
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                if x[j, h] > ub[h] or x[j, h] < lb[h]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h] = x[sel, h]
                    else:
                        x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment (relative to x[0] as in your original code)
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# Fuel cost calculation
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP)))
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []  # kept name per your original code
summary_rows = []  # NEW: clean summary rows for Excel

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    # Keep all_outputs for later ramp-rate computation (don't try to write it directly to Excel)
    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P),
        'Total Fuel Cost': total_Fuel_Cost,
        'Computation Time': end_time - start_time
    })

    # Clean summary row (serializable)
    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp)': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': iterations
    })

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # kept as-is (total cost assigned to each unit)
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# Save results to Excel (write clean tables only)
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED) 6u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED) 6u.xlsx'")

Load Counter: 1
Output for Power Demand 1263.00 MW
   Unit  Power Produced (MW)
0     1           423.667528
1     2           188.296461
2     3           265.870977
3     4           104.911859
4     5           172.400069
5     6           107.853106
Total Power Produced: 1263.00 MW
Total Fuel Cost: Rp 15290.55740487672
Computation Time: 0.037860870361328125 seconds
------------------------------------------------

All results saved to 'DOA (ED) 6u.xlsx'


In [13]:
import pandas as pd
import numpy as np
import random
import time

# Read input data
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/15-unit.xlsx',
    sheet_name='Table 1'
)
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd15u.xlsx')
power_demand = power_demand_data['load'].values

# Define Dream Optimization Algorithm_DOA-based power calculation
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    pop=30, max_iter=100):
    dim = len(pmin)
    lb, ub = pmin, pmax

    # Objective: Fuel cost + penalty for demand mismatch
    def fobj(pos):
        fuel = np.sum(a + b * pos + c * pos**2)
        penalty = abs(demand - np.sum(pos)) * 10000
        return fuel + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            # choose k with safe randint bounds
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))  # inclusive upper bound
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        # Boundary handling
                        if x[j, h] > ub[h] or x[j, h] < lb[h]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h] = x[sel, h]
                            else:
                                x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        # Local exploitation around sbest
        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            # safe randint when km could be 2
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                if x[j, h] > ub[h] or x[j, h] < lb[h]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h] = x[sel, h]
                    else:
                        x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment (relative to x[0] as in your original code)
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# Fuel cost calculation
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP)))
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []  # kept name per your original code
summary_rows = []  # NEW: clean summary rows for Excel

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    # Keep all_outputs for later ramp-rate computation (don't try to write it directly to Excel)
    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P),
        'Total Fuel Cost': total_Fuel_Cost,
        'Computation Time': end_time - start_time
    })

    # Clean summary row (serializable)
    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp)': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': iterations
    })

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # kept as-is (total cost assigned to each unit)
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# Save results to Excel (write clean tables only)
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED) 15u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED) 15u.xlsx'")

Load Counter: 1
Output for Power Demand 2630.00 MW
    Unit  Power Produced (MW)
0      1           358.742979
1      2           320.978830
2      3           120.577056
3      4            63.644980
4      5           345.960572
5      6           396.357776
6      7           273.376517
7      8           214.672729
8      9           159.066234
9     10           119.801489
10    11            75.095035
11    12            58.113718
12    13            56.996139
13    14            42.788536
14    15            23.827411
Total Power Produced: 2630.00 MW
Total Fuel Cost: Rp 32924.289993255145
Computation Time: 0.0357823371887207 seconds
------------------------------------------------

All results saved to 'DOA (ED) 15u.xlsx'


In [114]:
import pandas as pd
import numpy as np
import random
import time

# Read input data
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/40-unit.xlsx',
    sheet_name='Table 1'
)
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd40u.xlsx')
power_demand = power_demand_data['load'].values

# Define Dream Optimization Algorithm_DOA-based power calculation
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    pop=30, max_iter=100):
    dim = len(pmin)
    lb, ub = pmin, pmax

    # Objective: Fuel cost + penalty for demand mismatch
    def fobj(pos):
        fuel = np.sum(a + b * pos + c * pos**2)
        penalty = abs(demand - np.sum(pos)) * 10000
        return fuel + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            # choose k with safe randint bounds
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))  # inclusive upper bound
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        # Boundary handling
                        if x[j, h] > ub[h] or x[j, h] < lb[h]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h] = x[sel, h]
                            else:
                                x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        # Local exploitation around sbest
        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            # safe randint when km could be 2
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                if x[j, h] > ub[h] or x[j, h] < lb[h]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h] = x[sel, h]
                    else:
                        x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment (relative to x[0] as in your original code)
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# Fuel cost calculation
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP)))
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []  # kept name per your original code
summary_rows = []  # NEW: clean summary rows for Excel

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    # Keep all_outputs for later ramp-rate computation (don't try to write it directly to Excel)
    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P),
        'Total Fuel Cost': total_Fuel_Cost,
        'Computation Time': end_time - start_time
    })

    # Clean summary row (serializable)
    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp)': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': iterations
    })

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # kept as-is (total cost assigned to each unit)
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# Save results to Excel (write clean tables only)
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED) 40u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED) 40u.xlsx'")

Load Counter: 1
Output for Power Demand 10500.00 MW
    Unit  Power Produced (MW)
0      1           107.490687
1      2           111.291548
2      3           106.772866
3      4           108.773334
4      5            60.756320
5      6           106.408683
6      7           202.792828
7      8           287.622178
8      9           239.162981
9     10           266.060087
10    11           214.428402
11    12           331.503406
12    13           312.972435
13    14           417.103235
14    15           495.789578
15    16           395.975703
16    17           494.342495
17    18           474.152936
18    19           489.619010
19    20           442.728114
20    21           518.764562
21    22           533.850783
22    23           451.205609
23    24           375.655631
24    25           542.317088
25    26           436.418406
26    27           117.869626
27    28            27.549118
28    29           115.256138
29    30            48.843635
30    31          